# Embedding Space Analysis

Loads a sample of intent examples, projects them with the trained checkpoint, and inspects the embedding space with a silhouette score, 2D plots, and a nearest-neighbor sanity check.


In [ ]:
import json
import random
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from umap import UMAP

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.intent_embedder import ToolIntentEmbedder
from models.projection_head import ProjectionHead

DATA_PATH = ROOT / 'data/man/nl_command_pairs_flat_train_v2.jsonl'
CHECKPOINT_PATH = ROOT / 'checkpoints/intent_embedder/best_model.pt'
MAX_SAMPLES = 4000
BATCH_SIZE = 64
TOP_K_TOOLS = 20
SEED = 42
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'device: {DEVICE}')
print(f'data: {DATA_PATH}')
print(f'checkpoint: {CHECKPOINT_PATH}')


In [ ]:
def load_jsonl(path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]


def batched(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]


rows = load_jsonl(DATA_PATH)
random.shuffle(rows)
rows = rows[:MAX_SAMPLES]

tool_counts = Counter(row['tool'] for row in rows)
tool_names = sorted(tool_counts)
tool_to_idx = {tool: idx for idx, tool in enumerate(tool_names)}
labels = np.array([tool_to_idx[row['tool']] for row in rows])

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
cfg = ckpt.get('config', {})
dtype_name = cfg.get('torch_dtype', 'bfloat16')
dtype = {
    'float16': torch.float16,
    'bfloat16': torch.bfloat16,
    'float32': torch.float32,
}.get(dtype_name, torch.float32)

intent_embedder = ToolIntentEmbedder(
    model_name=cfg.get('encoder_model', 'Qwen/Qwen3.5-9B'),
    embedding_dim=cfg.get('intent_embedding_dim', 1024),
    pooling_strategy=cfg.get('pooling_strategy', 'attention'),
    dropout=cfg.get('dropout', 0.15),
    torch_dtype=dtype_name,
    max_length=cfg.get('max_length', 512),
).to(DEVICE)
intent_embedder.load_state_dict(ckpt['intent_embedder_state_dict'])
intent_embedder.eval()

projection_head = ProjectionHead(
    input_dim=cfg.get('intent_embedding_dim', 1024),
    output_dim=cfg.get('projection_dim', 128),
    dropout=cfg.get('dropout', 0.15),
).to(device=DEVICE, dtype=dtype)
projection_head.load_state_dict(ckpt['projection_head_state_dict'])
projection_head.eval()

tool_calls = [
    {'tool': row['tool'], 'arguments': {'command': row['command']}}
    for row in rows
]
queries = [row['nl_query'] for row in rows]

projected_batches = []
with torch.inference_mode():
    for tool_batch, query_batch in zip(
        batched(tool_calls, BATCH_SIZE),
        batched(queries, BATCH_SIZE),
    ):
        embeddings = intent_embedder(tool_calls=tool_batch, queries=query_batch)
        projected_batches.append(projection_head(embeddings).float().cpu())

proj = torch.cat(projected_batches).numpy()

print(f'samples: {len(rows)}')
print(f'unique tools: {len(tool_counts)}')
print(f'projected shape: {proj.shape}')
print('top tools:', tool_counts.most_common(10))


In [ ]:
def plot_projection(xy, title, xlabel, ylabel):
    top_tools = [tool for tool, _ in tool_counts.most_common(TOP_K_TOOLS)]

    plt.figure(figsize=(10, 8))
    for tool in top_tools:
        idx = [i for i, row in enumerate(rows) if row['tool'] == tool]
        plt.scatter(xy[idx, 0], xy[idx, 1], s=8, alpha=0.6, label=tool)

    plt.title(f'{title} (top {TOP_K_TOOLS} tools)')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()


if len(tool_counts) > 1:
    sil = silhouette_score(proj, labels, metric='cosine')
    print(f'silhouette (cosine): {sil:.4f}')
else:
    print('Not enough label variety for silhouette.')

plot_projection(
    PCA(n_components=2, random_state=SEED).fit_transform(proj),
    'PCA of projected intent embeddings',
    'PC1',
    'PC2',
)

plot_projection(
    UMAP(n_components=2, random_state=SEED).fit_transform(proj),
    'UMAP of projected intent embeddings',
    'UMAP1',
    'UMAP2',
)


In [ ]:
similarity = cosine_similarity(proj, proj)
np.fill_diagonal(similarity, -1.0)

for idx in random.sample(range(len(rows)), min(10, len(rows))):
    nn = int(np.argmax(similarity[idx]))
    print('---')
    print('Query:   ', rows[idx]['nl_query'])
    print('Tool:    ', rows[idx]['tool'])
    print('NN tool: ', rows[nn]['tool'])
    print('NN query:', rows[nn]['nl_query'])
